# Part 2: Using the `icepyx` python library to access ICESat-2 data

## Tutorial Overview

This tutorial is designed for the "[Cloud Computing and Open-Source Scientific Software for Cryosphere Communities](https://agu.confex.com/agu/fm23/meetingapp.cgi/Session/193477)" Learning Workshop at the 2023 AGU Fall Meeting.

This notebook demonstrates how to search for, access, and analyse and plot a cloud-hosted ICESat-2 dataset using the [`icepyx`](https://icepyx.readthedocs.io/en/latest/example_notebooks/IS2_data_access.html) package.

::::{admonition} System Requirements
:class: important
- **Memory:** ~500 MB required to run this tutorial
::::

<figure>
<center>
    <img src="https://icepyx.readthedocs.io/en/latest/_static/icepyx_v2_oval_orig_nobackgr.png" alt='icepyx logo of the word icepyx in raised letters on an iceberg with an ice ax'/>
</center>
</figure>

icepyx is a community and software library for searching, downloading, and reading ICESat-2 data. While opening data should be straightforward, there are some oddities in navigating the highly nested organization and hundreds of variables of the ICESat-2 data. icepyx provides tools to help with those oddities.

`icepyx` was started and initially developed by Jessica Scheick to provide easy programmatic access to ICESat-2 data (before `earthaccess` existed!) and facilitate collaborative development around ICESat-2 data products, including training, skill building, and support around practicing open science and contributing to open-source software. Thanks to contributions from countless community members, `icepyx` can (for ICESat-2 data): 
- search for available data granules (data files)
- order and download data or access it directly in the cloud
- order a subset of data: clipped in space, time, containing fewer variables, or a few other options provided by NSIDC
- search through the available ICESat-2 data variables
- read ICESat-2 data into xarray DataArrays, including merging data from multiple files

Under the hood, `icepyx` relies on `earthaccess` to help handle authentication, especially for obtaining S3 tokens to access ICESat-2 data in the cloud. All this happens without the user needing to take any action other than supplying their Earthdata Login credentials using one of the methods described in the `earthaccess` tutorial.

In this tutorial we will look at the `ATL08` Land and Vegetation Height product.


### Learning Objectives

In this tutorial you will learn:  
1. how to use `icepyx` to search for ICESat-2 data using spatial and temporal filters;  
2. how to open and combine data multiple HDF5 groups into an `xarray.Dataset` using `icepyx.Read`;  
3. how to begin your analysis, including selecting strong/weak beams and plotting.  

## Prerequisites

The workflow described in this tutorial forms the initial steps of an _Analysis in Place_ workflow that would be run on a AWS cloud compute resource.  You will need:

1. a JupyterHub, such as CryoHub, or AWS EC2 instance in the us-west-2 region.
3. a NASA Earthdata Login.  If you need to register for an Earthdata Login see the [Getting an Earthdata Login](https://icesat-2-2023.hackweek.io/preliminary/checklist/earthdata.html#getting-an-earthdata-login) section of the ICESat-2 Hackweek 2023 Jupyter Book.
4. A `.netrc` file, that contains your Earthdata Login credentials, in your home directory. See [Configure Programmatic Access to NASA Servers](https://icesat-2-2023.hackweek.io/preliminary/checklist/earthdata.html#configure-programmatic-access-to-nasa-servers) to create a `.netrc` file.

## Credits

This notebook is based on an [icepyx Tutorial](https://nasa-openscapes.github.io/2023-ssc/tutorials/data-access/icepyx.html) originally created by Rachel Wegener, Univ. Maryland and updated by Amy Steiker, NSIDC, and Jessica Scheick, Univ. of New Hampshire.
It was updated in May 2024 to utilize (at a minimum) v1.0.0 of icepyx.

## Using `icepyx` to search and access ICESat-2 data

We won't dive into using icepyx to search for and download data in this tutorial, since we already discussed how to do that with `earthaccess`. The code to search and download is still provided below for the curious reader. The [icepyx documentation](https://icepyx.readthedocs.io/en/latest/example_notebooks/IS2_data_access.html) shows more detail about different search parameters and how to inspect the results of a query.

In [ ]:
import icepyx as ipx

In [ ]:
import json
import math
import warnings

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from shapely.geometry import shape, GeometryCollection

In [ ]:
%matplotlib inline

We'll search for ATL08 tracks that intersect with the [La Primavera Biosphere Reserve](https://en.wikipedia.org/wiki/La_Primavera_Biosphere_Reserve), Jalisco, Mexico, located west of Guadalajara. 

In [ ]:
# Open a geojson of our area of interest
with open("./bosque_primavera.json") as f:
    features = json.load(f)["features"]

bosque = GeometryCollection([shape(feature["geometry"]).buffer(0) for feature in features])
bosque

In [ ]:
# Use our search parameters to setup a search Query
short_name = 'ATL08'
spatial_extent = list(bosque.bounds)
date_range = ['2019-05-04','2019-05-04']
region = ipx.Query(short_name, spatial_extent, date_range)

In [ ]:
# Display if any data files, or granules, matched our search
region.avail_granules(ids=True)

In [ ]:
# We can also get the S3 urls
print(region.avail_granules(ids=True, cloud=True))
s3urls = region.avail_granules(ids=True, cloud=True)[1]

In [ ]:
# Download the granules to a into a folder called 'bosque_primavera_ATL08'
region.order_granules()
region.download_granules('./bosque_primavera_ATL08')

<div class="alert alert-block alert-info">
<b>Tip:</b> If you don't want to type your Earthdata Login information every time they are
    required you can setup more automatic methods of authentication. Two common methods
    are 1) Add your earthdata password and username to as environment variables
    as EARTHDATA_USERNAME and EARTHDATA_PASSWORD. 2) setup a .netrc file in your home directory. See <a href="https://nasa-openscapes.github.io/2021-Cloud-Hackathon/tutorials/04_NASA_Earthdata_Authentication.html"> the Openscapes tutorial</a> </div>

## Reading a file with icepyx

To read a file with icepyx there are several steps:
1. Create a `Read` object. This sets up an initial connection to your file(s) and validates the metadata.
2. Tell the `Read` object what variables you would like to read
3. Load your data!

### Create a `Read` object

Here we are creating a read object to set up an initial connection to your file(s).

In [ ]:
# access the file you've downloaded
reader = ipx.Read('./bosque_primavera_ATL08')

In [ ]:
# access the file on the cloud (see "When to Cloud", below)
# reader = ipx.Read(s3urls)

In [ ]:
reader

### Select your variables

To view the variables contained in your dataset you can call `.vars` on your data reader.

In [ ]:
reader.variables.avail()

Thats **a lot** of variables!

One key feature of icepyx is the ability to browse the variables available in the dataset. There are typically hundreds of variables in a single dataset, so that is a lot to sort through! Let's take a moment to get oriented to the organization of ATL08 variables, by first a few important pieces of the algorithm.

To create higher level variables like canopy or terrain height, the ATL08 algorithms goes through a series of steps:
1. Identify signal photons from noise photons
2. Classify each of the signal photons as either terrain, canopy, or canopy top
3. Remove elevation, so the heights are with respect to the ground
3. Group the signal photons into 100m segments. If there are a sufficient number of photons in that group, calculate statistics for terrain and canopy (ex. mean height, max height, standard deviation, etc.)


![ATL08_photon_classification example](https://nasa-openscapes.github.io/2023-ssc/tutorials/data-access/.images/ATL08_photon_classification_example.jpg)

> _Fig. 4. An example of the classified photons produced from the ATL08 algorithm. Ground photons (red dots) are labeled as all photons falling within a point spread function distance of the estimated ground surface. The top of canopy photons (green dots) are photons that fall within a buffer distance from the upper canopy surface, and the photons that lie between the top of canopy surface and ground surface are labeled as canopy photons (blue dots)._ (Neuenschwander & Pitts, 2019)

Providing all the potentially useful information from all these processing steps results in a data file that looks like:

![ATL08 File structure](https://nasa-openscapes.github.io/2023-ssc/tutorials/data-access/.images/ATL08_structure.png)

Another way to visualize these structure is to download one file and open it using https://myhdf5.hdfgroup.org/. 

Further information about each one of the variables is available in the [Algorithm Theoretical Basis Document (ATBD)](https://icesat-2.gsfc.nasa.gov/sites/default/files/page_files/ICESat2_ATL08_ATBD_r006.pdf) for ATL08.

There is lots to explore in these variables, but we will move forward using a common ATL08 variable: `h_canopy`, or the "98% height of all the individual relative canopy heights (height above terrain)" (ATBD definition).

In [ ]:
reader.variables.append(var_list=['h_canopy', 'latitude', 'longitude'])

Note that adding variables is a required step before you can load the data.

### Load the data!

In [ ]:
ds = reader.load()
ds

Here we have an xarray Dataset, a common Python data structure for analysis. To visualize the data we can plot it using:

In [ ]:
ds.plot.scatter(x="longitude", y="latitude", hue="h_canopy")

Notice also that the data is shown for just our area of interest! That is because of icepyx's subsetting feature. You can find more details on this feature in the icepyx example gallery [here](https://icepyx.readthedocs.io/en/latest/example_notebooks/IS2_data_access2-subsetting.html). 

## When to Cloud

The astute user has by now noticed that in this tutorial we downloaded a granule to read in rather than directly reading it from an S3 bucket. Recall from the previous tutorial that reading a single group was a time intensive step and did not include multiple groups. Due to the way ICESat-2 data is stored on disk (because of the file format - it doesn't matter if it's a local disk or cloud disk), accessing the data within the file is really slow via the virtual file system. Several efforts are under way to help address this issue, and icepyx will implement them as soon as they are available. Current efforts include:
- storing ICESat-2 data in a cloud-optimized format
- reading data using the [h5coro](https://github.com/ICESat2-SlideRule/h5coro) library

Please let Amy, Jessica, or one of the workshop leads know if you're interested in joining any of these conversations (or telling us what issues you've encountered). We'd love to have your input and use case!

## Some example plots

To close, here are a few more examples of reading and visualizing ATL08 data.

### Example 1: View the photon classifications

In [ ]:
# Set up the data reader
reader = ipx.Read('./bosque_primavera_ATL08')

In [ ]:
# Add the photon height and classification variables
reader.variables.append(var_list=['ph_h', 'classed_pc_flag', 'latitude', 'longitude'])

In [ ]:
ds_photons = reader.load() 
ds_photons

In [ ]:
# Select just one beam.  `spot=6` corresponds to the right beam of the 3rd beam pair, gt3r.
gt3r = ds_photons.sel(spot=6)

We can create a simple plot of photon height `ph_h` above the ground.  ATL08 classifies photons as noise, ground, canopy and top of canopy.  This classification is accessed through `classed_pc_flag`.  We can use `xarray` to color photons by type using the `hue` keyword.

In [ ]:
# A less complex plot
fig, ax = plt.subplots()
fig.set_size_inches(15, 4)
    
gt3r.plot.scatter(ax=ax, x='delta_time', y='ph_h', hue='classed_pc_flag')

This kind of plot is great for a quick look at the data.  A neater plot can also be created by using matplotlib directly to modify the plot above.

We use `ListedColormap` to create a color map that is more appropriate for vegetation and add a legend, title and y-axis label.  The legend labels are read directly from the `flag_meanings` attribute of the `classed_pc_flag` variable.

In [ ]:
# Created a colormap for classified photons
cmap = ListedColormap(['lightgrey', 'saddlebrown', 'lightgreen', 'darkgreen'])

# Assign flag_meanings to labels to be used in legend
labels = gt3r.classed_pc_flag.attrs["flag_meanings"].split()

fig, ax = plt.subplots(figsize=(12,7))

m = gt3r.plot.scatter(x='delta_time', y='ph_h', hue='classed_pc_flag', 
                      s=20,  cmap=cmap,
                      add_legend=False, add_colorbar=False,
                      ax=ax)

handles, old_labels = m.legend_elements()  # Get the legend elements so that labels can be modified
ax.legend(handles, labels)

ax.set_title("Classified Photons")
ax.set_ylabel('Height above the ground (m)', size=12)

### Plot the canopy compared to the ground height

In [ ]:
# Remove our previous variables
reader.variables.remove(all=True)
# Add the next set of variables to the list
reader.variables.append(var_list=['h_te_best_fit', 'latitude', 'longitude'])

In [ ]:
# turn off warnings for load to not show an xarray warning that was resolved in the coming release of icepyx v1.0.0
with warnings.catch_warnings(record=True):
    # load the data
    ds_te = reader.load()
    
ds_te

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(12, 3)

# plot the canopy height above ground level
(ds.h_canopy + ds_te.h_te_best_fit).plot.scatter(ax=ax, x="delta_time", y="h_canopy") # orange

# plot the terrain values
ds_te.plot.scatter(ax=ax, x="delta_time", y="h_te_best_fit") # blue

## Summary 

In this notebook we explored the opening and rendering ATL08 data with icepyx. We saw that icepyx will subset our downloaded data to our area of interest and also allows us to download only the variables we need. The ATL08 data has a folder-like structure with many variables to choose from. We focused on `h_canopy` and showed additional examples using the raw photons and `h_te_best_fit` for the ground height.

More information about ATL08 or icepyx can be found in:
- The [icepyx documentation](https://icepyx.readthedocs.io/en/latest/)
- The [Algorithm Theoretical Basis Document (ATBD)](https://icesat-2.gsfc.nasa.gov/sites/default/files/page_files/ICESat2_ATL08_ATBD_r006.pdf)
- Neuenschwander et. al. 2019, Remote Sens. Env. [DOI](https://doi.org/10.1016/j.rse.2018.11.005)